<a href="https://colab.research.google.com/github/shoukk8-afk/CIFAR10-CNN/blob/main/CIFAR10%E2%91%A1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import transforms, datasets, models
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torch.optim import lr_scheduler

In [2]:
#学習済みの重みをダウンロード
model = models.resnet18(weights='DEFAULT')

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 174MB/s]


In [3]:
#今回は最終層以外固定で行うため最終層以外を更新しない設定にする
for param in model.parameters():
    param.requires_grad = False

#MLPの最後の層の出力数をCIFAR10用に10にする
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 10)

#オプティマイザに渡すためにrequires_grad=Trueのものを抽出
param_to_update = [param for param in model.parameters() if param.requires_grad == True]
optimizer = optim.Adam(param_to_update, lr=1e-3)

In [4]:
#GPUに設定
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [5]:
#resnet18に渡すようのtransform
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

In [6]:
cifar10 = datasets.CIFAR10(root='/content/', train=True, download=True, transform=transform)
cifar10_val = datasets.CIFAR10(root='/content/', train=False, download=True, transform=test_transform)
train_loader = DataLoader(cifar10, batch_size=1024, shuffle=True)
test_loader = DataLoader(cifar10_val, batch_size=1024, shuffle=True)

100%|██████████| 170M/170M [00:04<00:00, 37.0MB/s]


In [7]:
#訓練ループ(lr_scheduler込み)の構築
def training_loop(n_epochs, optimizer, model, loss_fn, train_loader, test_loader):
    #schedulerは5エボックごとに学習率を0.1倍させる
    scheduler = lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)
    for epoch in range(1, n_epochs + 1):
        total = 0
        correct = 0
        loss_train =  0.0
        model.train()
        for imgs, labels in train_loader:
            #imgsとlabelsをGPUに移す
            imgs = imgs.to(device=device)
            labels = labels.to(device=device)

            optimizer.zero_grad()
            outputs = model(imgs)

            loss = loss_fn(outputs, labels)
            loss.backward()
            optimizer.step()

            #1回のループの損失を加算して訓練データの1エボックの損失の合計を出す
            loss_train += loss.item()
        with torch.no_grad():
            model.eval()
            loss_val = 0.0
            for imgs, labels in test_loader:
                imgs = imgs.to(device=device)
                labels = labels.to(device=device)
                outputs = model(imgs)
                loss = loss_fn(outputs, labels)

                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

                #テストデータも同様に1エボックごとの損失の合計を出す
                loss_val += loss.item()

            accuracy = correct / total
            #学習率の更新
            scheduler.step()

        print(f"Epoch: {epoch}, Loss: {loss_train / len(train_loader)}")
        print(f"Accuracy: {accuracy}, Valloss: {loss_val / len(test_loader)}")

In [8]:
loss_fn = nn.CrossEntropyLoss()

In [9]:
#1～10エボック目
training_loop(10, optimizer, model, loss_fn, train_loader, test_loader)

Epoch: 1, Loss: 1.5749630149529905
Accuracy: 0.6983, Valloss: 1.040188193321228
Epoch: 2, Loss: 0.9467238783836365
Accuracy: 0.7435, Valloss: 0.814773565530777
Epoch: 3, Loss: 0.8048213732485868
Accuracy: 0.759, Valloss: 0.7389172196388245
Epoch: 4, Loss: 0.7431856612769925
Accuracy: 0.7699, Valloss: 0.6913009822368622
Epoch: 5, Loss: 0.7048707373288213
Accuracy: 0.7686, Valloss: 0.6777559518814087
Epoch: 6, Loss: 0.6822380399217411
Accuracy: 0.7783, Valloss: 0.6533109545707703
Epoch: 7, Loss: 0.6683563845498222
Accuracy: 0.7832, Valloss: 0.6356078386306763
Epoch: 8, Loss: 0.6516228074930153
Accuracy: 0.7868, Valloss: 0.6264936327934265
Epoch: 9, Loss: 0.6473473906517029
Accuracy: 0.788, Valloss: 0.6225620806217194
Epoch: 10, Loss: 0.638040345542285
Accuracy: 0.7882, Valloss: 0.6125200271606446


In [10]:
#API化のため重みを保存
torch.save(model.state_dict(), 'cifar10_resnet18.pth')